# Transit Ridership Cleaning Pipeline

نسخة مرتبة من البايبلاين مع عرض نتيجة كل خطوة باستخدام pandas و matplotlib.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

DATA_PATH = "transit_ridership.csv"
OUTPUT_DIR = Path("pipeline_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)


In [ ]:
def parse_mixed_date(value):
    if pd.isna(value):
        return pd.NaT

    value = str(value).strip()

    for fmt in ("%Y-%m-%d", "%m/%d/%Y", "%d-%b-%Y"):
        try:
            return pd.to_datetime(value, format=fmt)
        except ValueError:
            continue

    return pd.NaT


def impute_grouped_median(df, column, group_levels):
    for group_cols in group_levels:
        df[column] = df[column].fillna(
            df.groupby(group_cols)[column].transform("median")
        )
    df[column] = df[column].fillna(df[column].median())
    return df


## 1) Load raw data

In [ ]:
df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
print("Shape:", df.shape)
df.info()

In [ ]:
df.isna().sum().to_frame("missing_count")

## 2) Audit raw categories

In [ ]:
for col in ["route_id", "direction", "vehicle_type", "weather", "is_holiday"]:
    print(f"\n=== {col} ===")
    print(df[col].value_counts(dropna=False))


In [ ]:
plt.figure(figsize=(8, 4))
df["direction"].value_counts(dropna=False).plot(kind="bar")
plt.title("Raw direction values")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 3) Parse and standardize date

In [ ]:
df["date"] = df["date"].apply(parse_mixed_date)
print("date dtype:", df["date"].dtype)
print("Unparsed dates:", int(df["date"].isna().sum()))
df[["date"]].head(10)

In [ ]:
plt.figure(figsize=(8, 4))
df["date"].dt.month.value_counts().sort_index().plot(kind="bar")
plt.title("Rows per month after date parsing")
plt.xlabel("Month")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

## 4) Remove clearly invalid route IDs

In [ ]:
df = df[df["route_id"] != "R999"].copy()
df["route_id"].value_counts()

## 5) Standardize direction

In [ ]:
direction_map = {
    "Inbound": "Inbound",
    "INBOUND": "Inbound",
    "inbound": "Inbound",
    "In": "Inbound",
    "Inbnd": "Inbound",
    "Outbound": "Outbound",
    "OUTBOUND": "Outbound",
    "outbound": "Outbound",
    "Out": "Outbound",
    "Outbnd": "Outbound",
}

df["direction"] = (
    df["direction"]
    .astype(str)
    .str.strip()
    .replace(direction_map)
)

df["direction"].value_counts(dropna=False)

## 6) Standardize vehicle_type

In [ ]:
vehicle_type_map = {
    "Minibus": "Minibus",
    "MINIBUS": "Minibus",
    "mini bus": "Minibus",
    "Standard Bus": "Standard Bus",
    "standard bus": "Standard Bus",
    "Std Bus": "Standard Bus",
    "std bus": "Standard Bus",
    "Articulated Bus": "Articulated Bus",
    "articulated": "Articulated Bus",
}

df["vehicle_type"] = (
    df["vehicle_type"]
    .astype(str)
    .str.strip()
    .replace(vehicle_type_map)
)

df["vehicle_type"].value_counts(dropna=False)

## 7) Standardize is_holiday to boolean

In [ ]:
holiday_map = {
    "False": False,
    "false": False,
    "No": False,
    "0": False,
    "True": True,
    "true": True,
    "Yes": True,
    "1": True,
}

df["is_holiday"] = (
    df["is_holiday"]
    .astype(str)
    .str.strip()
    .map(holiday_map)
    .astype("boolean")
)

print(df["is_holiday"].dtype)
df["is_holiday"].value_counts(dropna=False)

## 8) Convert numeric columns

In [ ]:
numeric_cols = ["boarding_count", "alighting_count", "trip_duration_min", "temperature_c"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df[numeric_cols].dtypes

## 9) Remove exact duplicates

In [ ]:
print("Duplicate rows before:", int(df.duplicated().sum()))
df = df.drop_duplicates().copy()
print("Shape after drop_duplicates:", df.shape)

## 10) Fill missing boarding_count

In [ ]:
print("Missing before:", int(df["boarding_count"].isna().sum()))

df = impute_grouped_median(
    df,
    "boarding_count",
    [
        ["route_id", "direction", "vehicle_type", "is_holiday"],
        ["route_id", "direction"],
        ["route_id"],
    ],
)

print("Missing after:", int(df["boarding_count"].isna().sum()))
df[["route_id", "direction", "vehicle_type", "is_holiday", "boarding_count"]].head(10)

In [ ]:
plt.figure(figsize=(8, 4))
df["boarding_count"].plot(kind="hist", bins=30)
plt.title("boarding_count after imputation")
plt.tight_layout()
plt.show()

## 11) Fill missing alighting_count

In [ ]:
print("Missing before:", int(df["alighting_count"].isna().sum()))

df = impute_grouped_median(
    df,
    "alighting_count",
    [
        ["route_id", "direction", "vehicle_type", "is_holiday"],
        ["route_id", "direction"],
        ["route_id"],
    ],
)

print("Missing after:", int(df["alighting_count"].isna().sum()))
df[["route_id", "direction", "vehicle_type", "is_holiday", "alighting_count"]].head(10)

In [ ]:
plt.figure(figsize=(8, 4))
df["alighting_count"].plot(kind="hist", bins=30)
plt.title("alighting_count after imputation")
plt.tight_layout()
plt.show()

## 12) Clean invalid trip_duration_min and fill missing values

In [ ]:
print("Missing before:", int(df["trip_duration_min"].isna().sum()))

invalid_duration_mask = (df["trip_duration_min"] <= 0) | (df["trip_duration_min"] > 180)
print("Invalid duration rows:", int(invalid_duration_mask.sum()))

df.loc[invalid_duration_mask, "trip_duration_min"] = pd.NA

df = impute_grouped_median(
    df,
    "trip_duration_min",
    [
        ["route_id", "direction", "vehicle_type"],
        ["route_id", "direction"],
        ["route_id"],
    ],
)

print("Missing after:", int(df["trip_duration_min"].isna().sum()))
print("Any <= 0:", bool((df["trip_duration_min"] <= 0).any()))
print("Any > 180:", bool((df["trip_duration_min"] > 180).any()))

df[["route_id", "direction", "vehicle_type", "trip_duration_min"]].head(10)

In [ ]:
plt.figure(figsize=(8, 4))
df["trip_duration_min"].plot(kind="hist", bins=30)
plt.title("trip_duration_min after cleaning")
plt.tight_layout()
plt.show()

## 13) Fill missing temperature_c

In [ ]:
print("Missing before:", int(df["temperature_c"].isna().sum()))

df["month"] = df["date"].dt.month

df["temperature_c"] = df["temperature_c"].fillna(
    df.groupby(["date", "weather"])["temperature_c"].transform("median")
)
df["temperature_c"] = df["temperature_c"].fillna(
    df.groupby(["month", "weather"])["temperature_c"].transform("median")
)
df["temperature_c"] = df["temperature_c"].fillna(
    df.groupby("month")["temperature_c"].transform("median")
)
df["temperature_c"] = df["temperature_c"].fillna(df["temperature_c"].median())

print("Missing after:", int(df["temperature_c"].isna().sum()))
df[["date", "weather", "temperature_c"]].head(10)

In [ ]:
plt.figure(figsize=(8, 4))
df["temperature_c"].plot(kind="hist", bins=30)
plt.title("temperature_c after imputation")
plt.tight_layout()
plt.show()

## 14) Final validation

In [ ]:
df.isna().sum().to_frame("missing_after_cleaning")

In [ ]:
df.head(10)

## 15) Example final charts

In [ ]:
monthly_boarding = (
    df.assign(month_label=df["date"].dt.to_period("M").astype(str))
      .groupby("month_label")["boarding_count"]
      .sum()
      .sort_index()
)

plt.figure(figsize=(10, 4))
monthly_boarding.plot(kind="line", marker="o")
plt.title("Monthly total boarding_count")
plt.xlabel("Month")
plt.ylabel("Total boardings")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
route_boarding = (
    df.groupby("route_id")["boarding_count"]
      .sum()
      .sort_values(ascending=False)
)

plt.figure(figsize=(10, 4))
route_boarding.plot(kind="bar")
plt.title("Total boarding_count by route")
plt.xlabel("Route ID")
plt.ylabel("Total boardings")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 16) Save cleaned data and summary

In [ ]:
cleaned_df = df.drop(columns=["month"], errors="ignore").copy()

summary = {
    "row_count": int(len(cleaned_df)),
    "date_min": cleaned_df["date"].min().strftime("%Y-%m-%d") if cleaned_df["date"].notna().any() else None,
    "date_max": cleaned_df["date"].max().strftime("%Y-%m-%d") if cleaned_df["date"].notna().any() else None,
    "route_count": int(cleaned_df["route_id"].nunique(dropna=True)),
    "routes": sorted(cleaned_df["route_id"].dropna().unique().tolist()),
    "direction_counts": {str(k): int(v) for k, v in cleaned_df["direction"].value_counts(dropna=False).to_dict().items()},
    "vehicle_type_counts": {str(k): int(v) for k, v in cleaned_df["vehicle_type"].value_counts(dropna=False).to_dict().items()},
    "weather_counts": {str(k): int(v) for k, v in cleaned_df["weather"].value_counts(dropna=False).to_dict().items()},
    "holiday_counts": {str(k): int(v) for k, v in cleaned_df["is_holiday"].value_counts(dropna=False).to_dict().items()},
    "boarding_total": float(cleaned_df["boarding_count"].sum()),
    "alighting_total": float(cleaned_df["alighting_count"].sum()),
    "trip_duration_mean": float(cleaned_df["trip_duration_min"].mean()),
    "temperature_mean": float(cleaned_df["temperature_c"].mean()),
    "missing_after_cleaning": {k: int(v) for k, v in cleaned_df.isna().sum().to_dict().items()},
}

cleaned_df.to_csv(OUTPUT_DIR / "cleaned_transit_ridership.csv", index=False)

with open(OUTPUT_DIR / "summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("Saved files to:", OUTPUT_DIR.resolve())